In [3]:
import torch
import cv2
import numpy as np
from depth_anything_3.api import DepthAnything3

## materials used:

- Depth Anything 3 docs - https://github.com/ByteDance-Seed/Depth-Anything-3/blob/main/docs/API.md

- gsplat docs - https://docs.gsplat.studio/main/

- 3D Gaussian Splatting | 3DGS Implementation from Scratch in PyTorch-Only - https://www.youtube.com/watch?v=ZXmN2gWPHus

- https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/


- From Images to Semantic 3D Gaussian Splatting with Python - https://medium.com/data-science-collective/from-images-to-semantic-3d-gaussian-splatting-with-python-complete-guide-ff9d3d240847

In [ ]:
def load_model(model="depth-anything/da3-base"):
    # 1. Setup GPU device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2. Load the model directly from Hugging Face
    # Options: "depth-anything/da3-small", "depth-anything/da3-base", "depth-anything/da3-large"
    model = DepthAnything3.from_pretrained("depth-anything/da3-base")
    model = model.to(device)

    return model, device

model, device = load_model()

print('model loaded')

[INFO ] using MLP layer as FFN


model loaded


In [24]:
import os

def get_images_paths(image_folder):
    image_paths = []

    for n, entry in enumerate(os.scandir(image_folder)):  
        if entry.is_file():
            image_paths.append(entry.path)
    print(f"from folder - {image_folder} - {n} images")

    return image_paths

image_folder = '/home/maciej/dev/depth/car_img'
image_paths = get_images_paths(image_folder)

from folder - /home/maciej/dev/depth/car_img - 9 images


In [41]:
def run_model_inference(model, image_paths):
    # 4. Run inference (Returns a dictionary with depth and metadata)
    prediction = model.inference(
        image=image_paths,
    )

    # Extract raw depth data for your point cloud pipeline
    # Note: V3 outputs raw depth and confidence metadata
    depth_map = prediction.depth
    confidence_map = prediction.conf # Useful for filtering out noise

    return depth_map, confidence_map


depth_map, confidence_map = run_model_inference(model, image_paths)

[WARN ] Images in batch have different sizes [(336, 504), (336, 504), (322, 504), (322, 504), (350, 504), (350, 504), (308, 504), (364, 504), (336, 504), (350, 504)]; center-cropping all to smallest (308,504)
[INFO ] Processed Images Done taking 0.17540931701660156 seconds. Shape:  torch.Size([10, 3, 308, 504])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 20.597400188446045 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0005223751068115234 seconds


In [ ]:
def merge_point_clouds(prediction, confidence_threshold):
    


In [ ]:
import numpy as np
import open3d as o3d
from PIL import Image

image_path = '/home/maciej/dev/depth/422b1df6e7202cdef0cf8824c206138f.jpg'
image = Image.open(image_path)

# 1. Run model inference
prediction = model.inference(image=[image_path])

[INFO ] Processed Images Done taking 0.01645684242248535 seconds. Shape:  torch.Size([1, 3, 420, 504])
[INFO ] Model Forward Pass Done. Time: 3.0391924381256104 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0004973411560058594 seconds


# Create a point cloud from rgb and depth map
### - https://stackoverflow.com/questions/59590200/generate-point-cloud-from-depth-image
### - https://codereview.stackexchange.com/questions/79032/generating-a-3d-point-cloud

In [11]:
def create_point_cloud(depth_map, rgb_image, intrinsics, extrinsics, confidence_map, confidence_thresh=0.5):
    h, w = depth_map.shape

    # 1. Standard pinhole projection math (Creates 2D grids)
    fx, fy = intrinsics[0, 0], intrinsics[1, 1]
    cx, cy = intrinsics[0, 2], intrinsics[1, 2]

    u, v = np.meshgrid(np.arange(w), np.arange(h))

    x = (u - cx) * depth_map / fx
    y = (v - cy) * depth_map / fy
    z = depth_map

    # Grid shape: (H, W, 3)
    points_cam = np.stack([x, y, z], axis=-1)
    
    # Color grid shape: (H, W, 3) normalized between 0.0 and 1.0
    colors_grid = rgb_image.astype(np.float32) / 255.0

    # 2. THE THRESHOLD LOGIC
    # Create a 2D boolean mask where confidence is high AND depth is valid (not 0)
    valid_mask = (confidence_map > confidence_thresh) & (depth_map > 0)

    # 3. Flatten the grids into simple lists using the mask
    # This automatically converts the data from (H, W, 3) to (N, 3)
    points_cam_flat = points_cam[valid_mask]  # Shape: (N, 3)
    colors_flat = colors_grid[valid_mask]      # Shape: (N, 3)

    # 4. Transform from Camera Space to World Space
    # Extrinsics broken into Rotation (3x3) and Translation (3,)
    R = extrinsics[:3, :3]
    t = extrinsics[:3, 3]
    
    # Math correction: Shifting points from camera origin to world origin
    # For a flat list of points (N, 3), the matrix math looks like this:
    points_world = (points_cam_flat - t) @ R

    return points_world, colors_flat


In [12]:
import cv2

def match_resolutions(rgb_image, depth_map):
    """
    Resizes the RGB image to match the exact height and width of the depth map.
    """
    # Get the target dimensions from the depth map grid
    target_height, target_width = depth_map.shape
    
    # OpenCV expects (width, height) for the dsize parameter
    resized_rgb = cv2.resize(rgb_image, (target_width, target_height), interpolation=cv2.INTER_LINEAR)
    
    return resized_rgb

In [ ]:
# 2. Extract arrays
depth_map = prediction.depth[0]
confidence_map = prediction.conf[0] 
intrinsics = prediction.intrinsics[0]
extrinsics = prediction.extrinsics[0]

# 3. Load the raw image as a numpy array
rgb_image = np.array(image)

# 4. Resize the image to match the depth map
rgb_resized = match_resolutions(rgb_image, depth_map)

# 5. Call your manual point cloud function safely!
points_world, colors_flat = create_point_cloud(
    depth_map=depth_map,
    rgb_image=rgb_resized, 
    intrinsics=intrinsics,
    extrinsics=extrinsics,
    confidence_map=confidence_map,
    confidence_thresh=0.7
)

print(f"Points calculated! Array shape: {points_world.shape}")

[INFO ] Processed Images Done taking 0.021172523498535156 seconds. Shape:  torch.Size([1, 3, 420, 504])
[INFO ] Model Forward Pass Done. Time: 3.0738162994384766 seconds
[INFO ] Conversion to Prediction Done. Time: 0.00044798851013183594 seconds
Points calculated! Array shape: (211680, 3)


In [17]:
# 1. Instantiate an empty Open3D PointCloud object
pcd = o3d.geometry.PointCloud()

# 2. Assign your numpy coordinates (Vector3dVector converts them to Open3D format)
pcd.points = o3d.utility.Vector3dVector(points_world)

# 3. Assign your normalized colors
pcd.colors = o3d.utility.Vector3dVector(colors_flat)

# 4. Save to a standard .ply file
# Change this line in your script:
o3d.io.write_point_cloud("my_custom_point_cloud.ply", pcd, write_ascii=True)

print(f"Successfully saved {len(points_world)}")

Successfully saved 211680
